# 🐍 Python `def`: The Definitive & Complete Guide to Functions

A deep, production-grade guide covering everything about function definitions in Python: runtime execution & bytecode, parameter anatomy, scope resolution (LEGB), closures, first-class objects, introspection, and recursion.

---

## 📑 Table of Contents
1. [What is `def`? (Runtime Execution & Bytecode)](#1.-What-is-def%3F-(Runtime-Execution-&-Bytecode))
2. [Parameter & Argument Anatomy](#2.-Parameter-&-Argument-Anatomy)
   - 2.1 Positional vs Keyword Arguments
   - 2.2 Default Arguments
   - 2.3 🚨 The Mutable Default Argument Trap (`def f(lst=[])`)
   - 2.4 Positional-Only (`/`) & Keyword-Only (`*`)
   - 2.5 Variable Arguments (`*args` & `**kwargs`)
   - 2.6 The Canonical Signature Order
3. [Type Hints & Documentation (PEP 484 & Docstrings)](#3.-Type-Hints-&-Documentation-(PEP-484-&-Docstrings))
4. [Scope Resolution: The LEGB Rule](#4.-Scope-Resolution:-The-LEGB-Rule)
   - 4.1 Local, Enclosing, Global, Built-in
   - 4.2 `global` and `nonlocal` Modifiers
5. [First-Class Citizens & Design Patterns](#5.-First-Class-Citizens-&-Design-Patterns)
   - 5.1 Dispatch Table Pattern (Replacing if/elif chains)
   - 5.2 Higher-Order Functions
6. [Nested Functions & Closures](#6.-Nested-Functions-&-Closures)
   - 6.1 Closure Anatomy & Cell Variables
   - 6.2 Function Factories
7. [Introspection & Function Metadata](#7.-Introspection-&-Function-Metadata)
   - 7.1 Dunder Attributes (`__name__`, `__defaults__`, `__code__`)
   - 7.2 The `inspect` Module
8. [Recursion & Call Stack Limits](#8.-Recursion-&-Call-Stack-Limits)
9. [Summary Table & Best Practices](#9.-Summary-Table-&-Best-Practices)

## 1. What is `def`? (Runtime Execution & Bytecode)

### 💡 Concept
Unlike statically compiled languages (C/C++, Java) where functions are fixed declarations, in Python **`def` is an executable statement evaluated at runtime**.

When the Python interpreter executes a `def` statement:
1. It compiles the function body into an immutable **Code Object** (`code object`).
2. It instantiates a new `types.FunctionType` object on the heap.
3. It binds the function name identifier to this instance in the current namespace.

In [ ]:
import dis

# 1.1 Conditional function definition at runtime
PRODUCTION_MODE = False

if PRODUCTION_MODE:
    def process_data(data):
        return [x * 2 for x in data]
else:
    def process_data(data):
        print('[DEBUG] Processing with verbose checks')
        return [x * 2 for x in data]

print('Result:', process_data([10, 20, 30]))
print('Function type:', type(process_data))
print('Function name:', process_data.__name__)

# 1.2 Inspecting CPython bytecode disassembly
def add_numbers(a, b):
    return a + b

print('\n--- Disassembling Bytecode (dis.dis) ---')
dis.dis(add_numbers)

## 2. Parameter & Argument Anatomy

Python provides one of the most flexible parameter systems in modern programming: positional, keyword, defaults, restrictive delimiters (`/`, `*`), and variadic containers.

In [ ]:
# 2.1 Positional vs Keyword Arguments
def build_profile(username, role, organization='Freelance'):
    return f'{username} | {role} @ {organization}'

# Positional invocation (order matters)
print(build_profile('Lucas', 'Software Engineer', 'Google'))

# Keyword invocation (order is flexible)
print(build_profile(organization='DeepMind', role='AI Researcher', username='Lucas'))

In [ ]:
# 2.2 & 2.3 🚨 THE CLASSIC MUTABLE DEFAULT ARGUMENT TRAP
# Default parameter expressions are evaluated ONCE when the function is DEFINED, not on each call!

# ❌ DANGEROUS: The list instance is shared across all function calls!
def add_to_cart_bad(item, cart=[]):
    cart.append(item)
    return cart

print('Call 1 (Bad):', add_to_cart_bad('Keyboard'))
print('Call 2 (Bad):', add_to_cart_bad('Mouse'))  # ⚠️ 'Keyboard' is still in the cart!

# ✅ IDIOMATIC SOLUTION: Use None as a sentinel value and initialize inside
def add_to_cart_good(item, cart=None):
    if cart is None:
        cart = []
    cart.append(item)
    return cart

print('\nCall 1 (Good):', add_to_cart_good('Keyboard'))
print('Call 2 (Good):', add_to_cart_good('Mouse'))  # Fresh, independent list!

In [ ]:
# 2.4 Positional-Only (/) and Keyword-Only (*)
#   / -> Everything BEFORE must be passed purely positionally
#   * -> Everything AFTER must be passed as an explicit keyword

def configure_server(host, port, /, protocol='HTTP', *, timeout=30, enable_ssl=True):
    return {
        'url': f'{protocol.lower()}://{host}:{port}',
        'timeout': timeout,
        'ssl': enable_ssl
    }

# Valid call:
cfg = configure_server('127.0.0.1', 8080, 'HTTPS', timeout=15, enable_ssl=True)
print('Server Config:', cfg)

# 2.5 Packing & Unpacking (*args and **kwargs)
def data_pipeline(stage_name, *transforms, debug=False, **metadata):
    print(f'\n[Pipeline] Initial stage: {stage_name}')
    print(f'  *transforms tuple ({type(transforms).__name__}): {transforms}')
    print(f'  **metadata dict ({type(metadata).__name__}): {metadata}')

data_pipeline(
    'Ingestion',
    'Cleanse', 'Normalize', 'Aggregate',
    debug=True,
    author='Lucas',
    version='2.4.0'
)

### 2.6 The Canonical Signature Order

When combining all parameter types in Python, the syntax enforces this strict hierarchy:

$$\text{def func}(\text{pos\_only},\ /\ ,\ \text{standard\_pos\_or\_kw},\ \text{*args},\ \text{kw\_only},\ *\ ,\ \text{**kwargs}):$$

```python
def full_signature(a, b, /, c, d=10, *args, e, f=20, **kwargs):
    pass
```

## 3. Type Hints & Documentation (PEP 484 & Docstrings)

Modern Python leverages type hints for static analysis (Mypy, Pyright, IDE autocompletion) and structured docstrings (Google / NumPy styles).

In [ ]:
from typing import Optional, List, Callable

def process_transaction(
    amount: float,
    currency: str = 'USD',
    conversion_rate: Optional[float] = None,
    hooks: Optional[List[Callable[[float], None]]] = None
) -> float:
    """Calculates the net total of a financial transaction.

    Args:
        amount (float): The gross transaction value.
        currency (str, optional): The 3-letter currency code. Defaults to 'USD'.
        conversion_rate (Optional[float]): Exchange rate multiplier. Defaults to None (1.0).
        hooks (Optional[List[Callable]]): Post-process callback functions.

    Returns:
        float: The final converted amount.

    Raises:
        ValueError: If amount is negative.
    """
    if amount < 0:
        raise ValueError('Amount cannot be negative.')
    
    rate = conversion_rate if conversion_rate is not None else 1.0
    net_amount = amount * rate
    
    if hooks:
        for hook in hooks:
            hook(net_amount)
            
    return net_amount

print('Type Annotations stored in object:', process_transaction.__annotations__)
print('Docstring Preview:\n', process_transaction.__doc__.strip()[:160] + '...')

## 4. Scope Resolution: The LEGB Rule

Whenever an identifier is referenced inside a function, Python searches through 4 distinct scopes in sequential order:

```
 ┌─────────────────────────────────────────────────────────────┐
 │ L -> Local     : Variables assigned inside the function     │
 │ E -> Enclosing : Variables in outer enclosing functions     │
 │ G -> Global    : Module-level variables (top of file)       │
 │ B -> Built-in  : Built-in namespace (len, range, print, int)│
 └─────────────────────────────────────────────────────────────┘
```

In [ ]:
# 4.1 LEGB Hierarchy Demonstration
scope_level = 'GLOBAL (Module Level)'

def outer_function():
    scope_level = 'ENCLOSING (Outer Function)'
    
    def inner_function():
        scope_level = 'LOCAL (Inner Function)'
        print(f'1. Inner accesses: {scope_level}')
        
    inner_function()
    print(f'2. Outer accesses: {scope_level}')

outer_function()
print(f'3. Root accesses : {scope_level}')

In [ ]:
# 4.2 Mutating scopes with 'global' and 'nonlocal'
global_counter = 0

def increment_global():
    global global_counter
    global_counter += 10

increment_global()
print('Global counter after mutation:', global_counter)

# 'nonlocal' in nested functions (Stateful Closures)
def make_bank_account(initial_balance=0):
    balance = initial_balance  # Enclosing scope
    
    def deposit(amount):
        nonlocal balance  # Mutates enclosing 'balance' without polluting global
        balance += amount
        return balance
        
    return deposit

account = make_bank_account(100)
print('\nDeposit 50 -> Balance:', account(50))
print('Deposit 30 -> Balance:', account(30))

## 5. First-Class Citizens & Design Patterns

In Python, functions are first-class citizens. They can be assigned to variables, stored in data structures, passed into other functions, and returned from functions.

In [ ]:
# 5.1 Dispatch Table Pattern: Clean replacement for cascading if/elif/else
def cmd_save(data): return f'Saved: {data}'
def cmd_delete(data): return f'Deleted: {data}'
def cmd_export(data): return f'Exported {data} to JSON'

dispatcher = {
    'save': cmd_save,
    'delete': cmd_delete,
    'export': cmd_export
}

action = 'export'
handler = dispatcher.get(action, lambda d: 'Unknown command')
print('Dispatcher result:', handler('dataset_2026.parquet'))

# 5.2 Higher-Order Functions (Functions taking functions)
def apply_pipeline(numbers, transform_func):
    return [transform_func(n) for n in numbers]

def cube(x): return x ** 3

values = [1, 2, 3, 4]
print('\nHigher-order transform:', apply_pipeline(values, cube))

## 6. Nested Functions & Closures

A **Closure** occurs when an inner function remembers and has access to variables from its enclosing lexical scope, even after the outer function has finished executing.

In [ ]:
# Function Factory Pattern
def make_currency_formatter(symbol: str, precision: int = 2):
    # 'symbol' and 'precision' are captured inside the closure cell
    def format_price(amount: float) -> str:
        return f'{symbol} {amount:,.{precision}f}'
    return format_price

usd_format = make_currency_formatter('$', 2)
eur_format = make_currency_formatter('€', 2)
btc_format = make_currency_formatter('₿', 6)

print(usd_format(12450.75))
print(eur_format(12450.75))
print(btc_format(0.023412))

# Inspecting the closure cell objects
print('\nClosure cell objects:', usd_format.__closure__)
print('Captured values in cell:', [cell.cell_contents for cell in usd_format.__closure__])

## 7. Introspection & Function Metadata

Python functions carry rich metadata about their parameters, bytecode, and definitions, easily accessible via dunder attributes and the standard `inspect` module.

In [ ]:
import inspect

def api_endpoint(user_id: int, is_admin: bool = False, *scopes, timeout: int = 60, **options) -> dict:
    """Simulates an authenticated API endpoint handler."""
    return {'status': 'ok'}

# 7.1 Built-in Dunder Attributes
print('__name__        :', api_endpoint.__name__)
print('__defaults__    :', api_endpoint.__defaults__)    # Positional defaults
print('__kwdefaults__  :', api_endpoint.__kwdefaults__)  # Keyword-only defaults
print('__code__.co_varnames:', api_endpoint.__code__.co_varnames)

# 7.2 Programmatic Introspection via inspect.signature
sig = inspect.signature(api_endpoint)
print('\n--- Inspected Signature via inspect module ---')
print('Full signature:', sig)
for param_name, param in sig.parameters.items():
    print(f'  - Param: {param_name:<12} | Kind: {param.kind.name:<20} | Default: {param.default}')

## 8. Recursion & Call Stack Limits

Recursive functions call themselves to solve sub-problems. Every recursive function requires:
1. **Base Case:** A stopping condition that prevents infinite looping.
2. **Recursive Step:** A call that reduces the input closer to the base case.

In [ ]:
import sys

def factorial(n: int) -> int:
    # Base case
    if n <= 1:
        return 1
    # Recursive step
    return n * factorial(n - 1)

print('Factorial of 5:', factorial(5))
print('Factorial of 7:', factorial(7))

# Python's recursion limit guard against stack overflow
print(f'\nCurrent Python recursion limit: {sys.getrecursionlimit()} frames')

## 9. Summary Table & Best Practices

| Feature / Pattern | Syntax Example | Purpose | Pro Tip / Gotcha |
| :--- | :--- | :--- | :--- |
| **Standard Definition** | `def f(a, b):` | Declares a runtime callable | Use descriptive verbs in `snake_case` |
| **Safe Defaults** | `def f(lst=None):` | Fresh instances per call | 🚨 NEVER use mutable `lst=[]` as default |
| **Positional-Only** | `def f(a, /, b):` | `a` must be passed by position | Great for APIs when parameter names might change |
| **Keyword-Only** | `def f(a, *, b):` | `b` must be passed by keyword | Eliminates ambiguous boolean flags in calls |
| **`*args`** | `def f(*args):` | Accepts arbitrary positional args (Tuple) | Perfect for aggregation or wrapper functions |
| **`**kwargs`** | `def f(**kwargs):` | Accepts arbitrary keyword args (Dict) | Ideal for config pass-through & decorators |
| **Type Hints** | `def f(x: int) -> str:` | Declares explicit type contracts | Enables IDE autocomplete & Mypy safety |
| **`nonlocal`** | `nonlocal x` | Mutates enclosing function's variable | Enables stateful closures without globals |
| **`global`** | `global x` | Mutates module-level variable | Avoid in production to prevent hidden coupling |
| **Introspection** | `inspect.signature(f)` | Inspects parameters & types dynamically | Foundational for frameworks like FastAPI & Pydantic |